In [ ]:
import pandas as pd

# === Step 1: Load both CSV files ===
testing_file = "/home/dan_pham/Public/NIPTorrent/testing_samples.csv"
metadata_file = "/home/dan_pham/Public/NIPTorrent/METADATA1.csv"

testing = pd.read_csv(testing_file)
metadata = pd.read_csv(metadata_file)

# === Step 2: Standardize column names to avoid mismatch ===
testing.columns = testing.columns.str.strip()
metadata.columns = metadata.columns.str.strip()

# === Step 3: Merge based on shared identifiers ===
merged = pd.merge(
    testing,
    metadata[
        [
            "NGS RUN",
            "SAMPLE ID",
            "IONXPRESS BARCODE",
            "MATERNAL AGE",
            "GA",
            "FF (%)",
            "UNIQUE READS (M)",
            "Z21",
            "Z18",
            "Z13",
            "GENDER",
            "AVERAGE READ LENGTH (bp)",
            "GC CONTENT (%)",
            "DUPLICATION (%)",
            "Nuchal Translucency (mm)",
        ]
    ],
    on=["NGS RUN", "SAMPLE ID", "IONXPRESS BARCODE"],
    how="left",
    suffixes=("", "_meta"),
)

# === Step 4: Fill missing columns in testing_samples with metadata values ===
for col in [
    "MATERNAL AGE",
    "GA",
    "FF (%)",
    "UNIQUE READS (M)",
    "Z21",
    "Z18",
    "Z13",
    "GENDER",
    "AVERAGE READ LENGTH (bp)",
    "GC CONTENT (%)",
    "DUPLICATION (%)",
    "Nuchal Translucency (mm)",
]:
    merged[col] = merged[col].combine_first(merged[f"{col}_meta"])

# === Step 5: Drop extra columns ===
merged = merged[
    [c for c in merged.columns if not c.endswith("_meta")]
]

# === Step 6: Save output ===
output_path = r"C:\Users\ariha.danph\Downloads\NIPTorrent\testing_samples_filled.csv"
merged.to_csv(output_path, index=False)

print(f"✅ Filled data saved to: {output_path}")


✅ Filled data saved to: C:\Users\ariha.danph\Downloads\NIPTorrent\testing_samples_filled.csv


In [ ]:
import pandas as pd

# === Step 1: Load file ===
file_path = "/home/dan_pham/Public/NIPTorrent/test.csv"
df = pd.read_csv(file_path)

# === Step 2: Extract unique values ===
col1 = set(df['sample_test1'].dropna().astype(str).str.strip())
col2 = set(df['sample_test2'].dropna().astype(str).str.strip())

# === Step 3: Compare and find differences ===
to_delete = sorted(list(col1 - col2))  # in col1 but not in col2
to_add = sorted(list(col2 - col1))      # in col2 but not in col1

# === Step 4: Save to CSV files ===
pd.DataFrame({'to_delete': to_delete}).to_csv("/home/dan_pham/Public/NIPTorrent/delete.csv", index=False)
pd.DataFrame({'to_add': to_add}).to_csv("/home/dan_pham/Public/NIPTorrent/add.csv", index=False)

print("✅ Done!")
print(f"Samples in col1 but not in col2: {len(to_delete)} → delete.csv")
print(f"Samples in col2 but not in col1: {len(to_add)} → add.csv")


✅ Done!
Samples in col1 but not in col2: 1 → delete.csv
Samples in col2 but not in col1: 90 → add.csv


In [ ]:
library(readr)
library(dplyr)

# File paths
gold_file <- "/home/dan_pham/Public/NIPTorrent/testing_samples.csv"
result_file <- "/home/dan_pham/Downloads/RESULTS/sample_run/nipt_risk_without_blacklist/nipt_risk/results.csv"

# Output path
output_file <- "/home/dan_pham/Public/NIPTorrent/testing_samples_matched.csv"

# Read files
gold <- read_csv(gold_file, show_col_types = FALSE)
result <- read_csv(result_file, show_col_types = FALSE)

# Column names
gold_id <- names(gold)[1]
result_id <- names(result)[1]

# Match by sample names
matched <- gold %>%
    filter(.data[[gold_id]] %in% result[[result_id]])

# Save output
write_csv(matched, output_file)

cat("Done! Matched rows saved to:", output_file, "\n")


# LOAD DATA GENDER IN METADATA TO TABLE OF REFERENCE

In [4]:
import pandas as pd

# === Step 1: Load files ===
gender_file = "/home/dan_pham/Public/NIPT_data/reference/REF_100/TRIM_15_50/ref_sample/gender_prediction/compare_gender.csv"
metadata_file = "/home/dan_pham/Public/NIPTorrent/STATISTIC/DATA/METADATA1.csv"

gender = pd.read_csv(gender_file)
metadata = pd.read_csv(metadata_file)

# === Step 2: Clean column names ===
gender.columns = gender.columns.str.strip()
metadata.columns = metadata.columns.str.strip()

# === Step 3: Extract SAMPLE ID + BARCODE from GENDER.csv ===
# sample format = 24PC1151_021
gender["SAMPLE ID"] = gender["sample"].str.split("_").str[0]
gender["BARCODE_NUM"] = gender["sample"].str.split("_").str[1]
gender["IONXPRESS BARCODE"] = "IonXpress_" + gender["BARCODE_NUM"]

# === Step 4: Select metadata columns ===
meta_subset = metadata[["SAMPLE ID", "IONXPRESS BARCODE", "GENDER"]]

# === Step 5: Merge ===
merged = pd.merge(
    gender,
    meta_subset,
    on=["SAMPLE ID", "IONXPRESS BARCODE"],
    how="left",
    suffixes=("", "_meta"),
)

# === Step 6: Fill missing gender ===
merged["GENDER"] = merged["GENDER"].combine_first(merged["GENDER_meta"])

# If Gender column (F/M) is empty, fill based on metadata's XX/XY
merged.loc[merged["Gender"].isna() & merged["GENDER"].notna(), "Gender"] = \
    merged["GENDER"].map({"XX": "F", "XY": "M"})

# === Step 7: Drop helper columns ===
merged = merged.drop(columns=["BARCODE_NUM", "GENDER_meta"])

# === Step 8: Save output ===
merged.to_csv(gender_file, index=False)

print("✅ Completed! GENDER.csv updated with metadata gender values.")


✅ Completed! GENDER.csv updated with metadata gender values.
